# AI Insurance Data Analyst V2 — notebook pédagogique

Ce notebook explique bloc par bloc le code de l’application et son lien avec l’orchestration Agentic AI.

Principe directeur : **l’IA comprend l’intention et explique ; les services déterministes calculent et modifient les données**.

Parcours : `CONNECT → PROFILE → CLEAN → VALIDATE → ANALYZE → EXPLAIN → VISUALIZE → REPORT → EXPORT`.


## 1. Architecture globale

```text
Utilisateur → Streamlit/app.py → SupervisorAgent/AnalysisAgent
                                      │
        ┌─────────────────────────────┴─────────────────────────────┐
        │ Ingestion │ Profiling │ Cleaning │ Analytics │ KPI │ Report │
        └─────────────────────────────┬─────────────────────────────┘
                                      ▼
                           ProjectStore + AuditLogger

GeminiModelManager : planification JSON et explication, jamais calcul financier.
```

Une boucle agentique observe l’état, planifie, choisit un outil autorisé, exécute, vérifie et journalise. Elle ne lance pas de Python arbitraire.


In [ ]:
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'insurance_ai').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'sample_data' / 'insurance_sample.csv'
df = pd.read_csv(DATA_PATH)
print(len(df), 'lignes |', len(df.columns), 'colonnes')
print('doublons =', int(df.duplicated().sum()), '| NA =', int(df.isna().sum().sum()))
df.head()


## 2. Configuration et initialisation de `app.py`

`core/config.py` construit `Settings` à partir des variables d’environnement : répertoires, taille maximale d’upload, clé Gemini, modèles autorisés, retries et timeout. Les secrets ne sont jamais codés dans l’application.

`app.py` instancie `ProjectStore`, `AuditLogger`, `IngestionService`, `ProfilingService`, `CleaningService`, `DashboardService`, `ReportGenerator`, etc. `st.session_state` conserve utilisateur, projet, dataset actif et résultats entre deux reruns.

Le rechargement défensif des services traite les processus Streamlit qui gardent un ancien module Python en mémoire.


In [ ]:
import os
print('APP_ENV =', os.getenv('APP_ENV', 'development'))
print('Gemini configuré =', bool(os.getenv('GOOGLE_API_KEY')))
print('Limite upload =', os.getenv('MAX_UPLOAD_MB', '200'), 'MB')


## 3. Ingestion et prévisualisation RAW

`IngestionService` détecte encodage, séparateur et décimale des CSV, liste les feuilles Excel et aplati les JSON tabulaires. La page Importer contrôle la taille, affiche le parsing puis appelle `save_version(..., stage='RAW')`.

L’aperçu RAW est conservé dans la session pour rester visible après un rerun ou un nettoyage STAGING. RAW est la copie de la source et ne doit pas être écrasé.


In [ ]:
from insurance_ai.services.ingestion import IngestionService
ingestion = IngestionService()
detection = ingestion.detect_csv(DATA_PATH.read_bytes())
loaded, info = ingestion.read_uploaded(DATA_PATH.name, DATA_PATH.read_bytes())
print(detection)
print(info)
loaded.head(3)


## 4. Profilage et qualité

`ProfilingService.profile(df)` calcule dimensions, types, types sémantiques, NA, cardinalité, exemples, min/max/moyenne/médiane et score qualité. Il ne modifie pas le DataFrame.

Les règles métier (`not_null`, `unique`, `non_negative`, `allowed_values`, `date_order`) évaluent les violations et peuvent alimenter un rapport.


In [ ]:
from insurance_ai.services.profiling import ProfilingService
profile = ProfilingService().profile(df)
print(profile.summary)
print('doublons =', profile.duplicate_rows)
profile.columns[['column', 'missing', 'missing_pct', 'semantic_type']]


## 5. Nettoyage et imputation cellule par cellule

`CleaningService` expose normalisation, déduplication, cast, catégories, filtres et imputation. L’UI liste dynamiquement les variables avec NA, sélectionne une ligne, choisit une stratégie puis appelle `impute_cell`. Après chaque action, Streamlit rerun, recalcule les NA et écrit un `TransformationRecord`.

Les identifiants potentiels ne peuvent recevoir qu’une valeur manuelle vérifiée : une moyenne ne doit jamais inventer une clé de police ou de sinistre.


In [ ]:
from insurance_ai.services.cleaning import CleaningService
cleaner = CleaningService()
positions = [i for i, missing in enumerate(df['region'].isna()) if missing]
outcome = cleaner.impute_cell(df, 'region', positions[0], 'manual', 'Abidjan')
print(outcome.details)
print('NA avant/après =', df.region.isna().sum(), outcome.dataframe.region.isna().sum())


## 6. Versions RAW / STAGING / CURATED

`ProjectStore` écrit les datasets dans un projet et conserve source, stade, parent, dimensions, empreinte et chemin relatif. RAW conserve la source ; STAGING porte les transformations ; CURATED est la version validée et autorisée pour les analyses.


In [ ]:
from tempfile import TemporaryDirectory
from insurance_ai.core.project_store import ProjectStore
with TemporaryDirectory() as folder:
    store = ProjectStore(folder)
    project_id = store.create_project('Notebook', owner='demo')
    raw_meta = store.save_dataframe(project_id, df, name='sample', stage='RAW', source_type='csv', source_name=DATA_PATH.name)
    print(raw_meta.to_dict())


## 7. KPI assurance déterministes

`InsuranceKPIService` calcule S/P, coût moyen, fréquence, encaissement, cession et rétention. Les résultats contiennent valeur, formule et avertissements ; les dénominateurs nuls sont traités. Gemini ne produit jamais ces chiffres.


In [ ]:
from insurance_ai.services.insurance_kpi import InsuranceKPIService
kpis = InsuranceKPIService()
loss_ratio = kpis.loss_ratio(df, 'claim_amount', 'written_premium')
print(loss_ratio)
print(kpis.as_display(loss_ratio))


## 8. Dashboard sur RAW et CURATED

`DashboardService.metric_columns` filtre selon le sens métier : primes (`written_premium`, `collected_premium`, `ceded_premium`, `net_premium`) et sinistres (`claim_amount`, coûts ou montants). Les identifiants sont exclus.

Le même service reçoit le DataFrame RAW ou CURATED actif et produit cartes KPI, segments, tendances et principaux sinistres.


In [ ]:
from insurance_ai.services.dashboard import DashboardService
dashboard = DashboardService()
premium_cols = dashboard.metric_columns(df, 'premium')
claim_cols = dashboard.metric_columns(df, 'claims')
print('primes =', premium_cols)
print('sinistres =', claim_cols)
view = dashboard.portfolio_overview(df, premium_col=premium_cols[0], claims_col=claim_cols[0], group_col='branch', date_col='effective_date')
view.cards


## 9. Analytics allow-listées

`AnalyticsService.OPS` autorise seulement description, comptage, agrégations, top valeurs et corrélations. Les filtres sont limités à `==`, `!=`, `>`, `>=`, `<`, `<=`, `contains`.

Même si Gemini propose un plan invalide, `_validate_plan` le rejette avant exécution. Aucun `eval()`/`exec()` n’est utilisé.


In [ ]:
from insurance_ai.services.analytics import AnalyticsService
analytics = AnalyticsService()
result = analytics.execute(df, {'operation': 'mean', 'column': 'written_premium', 'filters': []})
print(result.answer)
print(result.calculation)


## 10. SupervisorAgent et AnalysisAgent

`SupervisorAgent.route()` choisit profilage, nettoyage ou analyse. `AnalysisAgent` résume le schéma, demande éventuellement à Gemini un plan JSON, valide opération/colonnes/filtres puis délègue à `AnalyticsService`.

Sans clé Gemini, une heuristique offline reconnaît moyenne, somme, corrélation, top valeurs et description. Avec Gemini, seul le plan est généré ; Pandas calcule.


In [ ]:
from insurance_ai.agents.supervisor import SupervisorAgent
supervisor = SupervisorAgent()
print(supervisor.route('analyze'))
answer, plan = supervisor.ask('Quelle est la moyenne de written_premium ?', df)
print('plan =', plan)
print('résultat =', answer.data)
print('calcul =', answer.calculation)


## 11. GeminiModelManager : fallback et gouvernance

Le gestionnaire centralise timeout, retries avec backoff, modèles de secours et latence. `generate_json` parse les plans. Si tous les modèles échouent, qualité, nettoyage, KPI, dashboard et reporting restent disponibles.

Le prompt reçoit un résumé de schéma plutôt que le dataset complet ; l’explication reçoit un résultat compact et doit mentionner la méthode sans inventer de nombres.


## 12. Reporting, audit et sécurité

`ReportPayload` alimente les rendus HTML/PDF/DOCX/PPTX/XLSX. `AuditLogger` journalise transformations, analyses et exports dans SQLite avec requêtes paramétrées.

Garde-fous : secrets hors Git, uploads limités, SQL lecture seule, chemins confinés, RAW préservé, identifiants protégés, opérations NL allow-listées et séparation logique des projets.


In [ ]:
from insurance_ai.core.models import TransformationRecord
record = TransformationRecord(transformation_id='demo-001', dataset_id='raw-demo', operation='impute_cell', column='region', parameters={'row_position': 9, 'strategy': 'manual', 'value': 'Abidjan'}, rows_before=len(df), rows_after=len(df), user='demo')
record.to_dict()


## 13. Flux complet Agentic AI

```text
Question → observation du schéma/session → plan JSON
        → validation allow-list → outil Pandas/KPI/dashboard
        → résultat déterministe → explication Gemini
        → audit + ajout éventuel au rapport
```

La valeur agentique vient de la coordination des outils et de la gestion de l’état. La sécurité vient de la validation avant l’action. Une évolution possible est de séparer des agents Ingestion, Qualité, Assurance, Visualisation et Reporting en conservant les mêmes contrats déterministes.


## Références du dépôt

- `app.py` : UI Streamlit, session et parcours utilisateur
- `insurance_ai/agents/analysis_agent.py` : planification/explanation NL
- `insurance_ai/agents/supervisor.py` : routage
- `insurance_ai/services/analytics.py` : opérations allow-listées
- `insurance_ai/services/gemini_manager.py` : fallback Gemini
- `insurance_ai/core/project_store.py` : versions RAW/STAGING/CURATED
- `insurance_ai/core/audit.py` : audit SQLite
- `docs/ARCHITECTURE.md`, `docs/SECURITY.md`, `docs/MOA_TRACEABILITY.md` : documentation de référence
